In [1]:
import logging
import pathlib

import mcr_py.helper_functions
import mcr_py.mcr5.labels
import mcr_py.minute_city.minute_city
import polars as pl
from mcr_py.utils.logger import setup

setup("INFO")

In [2]:
city_name = "cologne"
date = "20250926"

In [3]:
data_directory = pathlib.Path("../data/")
base_directory = data_directory / date
cache_path = base_directory / "cache/"
osm_path = base_directory / "osm_raw"
geometa_path = base_directory / f"cache/{city_name}_geometa.pkl"
mcr5_output_path = base_directory / f"mcr5_results/{city_name}"
geo_meta, geo_data = mcr_py.helper_functions.load_auxiliary_classes(
    geo_meta_path=geometa_path,
    city_id="Koeln",
    osm_path=osm_path,
    cache_path=cache_path,
)

[14:49:35] INFO     Loading OSM walking                               ]8;id=446299;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=768094;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#62\62]8;;\
[14:49:38] INFO     Loading OSM walking done (2.64 seconds)           ]8;id=256656;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=885564;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#62\62]8;;\
           INFO     Loading OSM POIs                                  ]8;id=428490;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=518218;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#69\69]8;;\
           INFO     Loading OSM POIs done (0.01 seconds)              ]8;id=379022;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=77568;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#69\69]8;;\
           INFO     Loadi

In [4]:
all_files = list((mcr5_output_path / "walking").iterdir())
logging.info(len(all_files))

           INFO     5344                                         ]8;id=640795;file:///tmp/ipykernel_502392/3566842583.py\3566842583.py]8;;\:]8;id=591300;file:///tmp/ipykernel_502392/3566842583.py#2\2]8;;\


In [5]:
hex_test = mcr_py.mcr5.labels.read_labels_for_nodes(
    str(mcr5_output_path / "public_transport_0"),
    geo_data.pois.with_columns(pl.col("nearest_osm_node").alias("osm_node_id")).lazy(),
)
hex_test = mcr_py.minute_city.minute_city.add_pois_to_labels(
    hex_test.lazy(), geo_data.pois.lazy()
)
hex_test = hex_test.collect()


hex_test_comp = mcr_py.mcr5.labels.read_labels_for_nodes(
    str(pathlib.Path("../data/mcr5/cologne_20240423") / "walking"),
    geo_data.pois.with_columns(pl.col("nearest_osm_node").alias("osm_node_id")).lazy(),
)
hex_test_comp = mcr_py.minute_city.minute_city.add_pois_to_labels(
    hex_test_comp.lazy(), geo_data.pois.lazy()
)
hex_test_comp = hex_test_comp.collect()

In [25]:
poi_types = geo_data.pois.get_column("poi_type").unique().to_list()

In [27]:
hex_test.filter(pl.col("time") == pl.col("time").max().over("start_id_hex")).unique(
    ["start_id_hex", "poi_type", "time"] + poi_types
).sum()

osm_node_id,time,cost,n_transfers,human_readable_time,start_id_hex,osm_id,lat,long,nearest_osm_node,dist_to_nearest,poi_type,target_id_osm,Grocery,Shops,Parks,Sustenance,Health,Banks,Education
i64,i64,i64,i64,str,str,u64,f64,f64,u64,f64,str,i64,i64,i64,i64,i64,i64,i64,i64
25461864667853,241103502,0,0,null,null,34680362309109,407002.184086,55617.350933,25461864667853,151760.566152,null,25461864667853,1292,1643,2300,1219,537,431,566


In [ ]:
hex_test_comp.join(hex_test, on=["start_id_hex", "poi_type", "osm_node_id"]).unique(
    ["start_id_hex", "poi_type", "osm_node_id", "time", "time_right"]
).filter((pl.col("time") - pl.col("time_right")).abs() > 180)

osm_node_id,time,cost,n_transfers,human_readable_time,start_id_hex,osm_id,lat,long,nearest_osm_node,dist_to_nearest,poi_type,target_id_osm,Education,Sustenance,Health,Shops,Grocery,Parks,Banks,time_right,cost_right,n_transfers_right,human_readable_time_right,osm_id_right,lat_right,long_right,nearest_osm_node_right,dist_to_nearest_right,target_id_osm_right,Health_right,Shops_right,Grocery_right,Parks_right,Banks_right,Sustenance_right,Education_right
i64,i64,i64,i64,str,str,u64,f64,f64,u64,f64,str,i64,i8,i8,i8,i8,i8,i8,i8,i64,i64,i64,str,u64,f64,f64,u64,f64,i64,i8,i8,i8,i8,i8,i8,i8
286806461,29287,0,0,"""08:08:07""","""891fa181627ffff""",1749096466,50.971366,7.0673151,286806461,30.564712,"""Parks""",286806461,0,0,0,0,0,1,0,29486,0,0,"""08:11:26""",1749096466,50.971366,7.0673151,286806461,30.564712,286806461,0,0,0,1,0,0,0
2093837701,29231,0,0,"""08:07:11""","""891fa199d3bffff""",307181210,50.956555,6.9556712,2093837701,15.301367,"""Education""",2093837701,1,0,0,0,0,0,0,29043,0,0,"""08:04:03""",307181210,50.956555,6.9556712,2093837701,15.301367,2093837701,0,0,0,0,0,0,1
928177774,31185,0,0,"""08:39:45""","""891fa19a85bffff""",1610945950,51.028724,6.9126644,928177774,11.269642,"""Parks""",928177774,0,0,0,0,0,1,0,31420,0,0,"""08:43:40""",1610945950,51.028724,6.9126644,928177774,11.269642,928177774,0,0,0,1,0,0,0
10591575910,29440,0,0,"""08:10:40""","""891fa198437ffff""",4147527410,50.998082,6.902092,10591575910,4.745025,"""Shops""",10591575910,0,0,0,1,0,0,0,29652,0,0,"""08:14:12""",4147527410,50.998082,6.902092,10591575910,4.745025,10591575910,0,1,0,0,0,0,0
1610945043,31364,0,0,"""08:42:44""","""891fa19a8cfffff""",2404152397,51.018804,6.9232041,1610945043,6.744666,"""Parks""",1610945043,0,0,0,0,0,1,0,31159,0,0,"""08:39:19""",2404152397,51.018804,6.9232041,1610945043,6.744666,1610945043,0,0,0,1,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6584537420,31232,0,0,"""08:40:32""","""891fa18a84bffff""",8820745326,50.896009,7.0739218,6584537420,12.016511,"""Shops""",6584537420,0,0,0,1,0,0,0,30978,0,0,"""08:36:18""",8820745326,50.896009,7.0739218,6584537420,12.016511,6584537420,0,1,0,0,0,0,0
8233800007,29923,0,0,"""08:18:43""","""891fa1983b3ffff""",290944624,50.979721,6.9480979,8233800007,31.183465,"""Sustenance""",8233800007,0,1,0,0,0,0,0,30157,0,0,"""08:22:37""",290944624,50.979721,6.9480979,8233800007,31.183465,8233800007,0,0,0,0,0,1,0
8507116653,29598,0,0,"""08:13:18""","""891fa18a84bffff""",12920666103,50.920421,7.0820206,8507116653,3.946434,"""Sustenance""",8507116653,0,1,0,0,0,0,0,29927,0,0,"""08:18:47""",12920666103,50.920421,7.0820206,8507116653,3.946434,8507116653,0,0,0,0,0,1,0


In [6]:
selected = hex_test.filter(pl.col("start_id_hex") == "881fa18ab5fffff")

In [7]:
poi_types = geo_data.pois.get_column("poi_type").unique().to_list()

In [8]:
import mcr_py.minute_city.profile

profiles = mcr_py.minute_city.profile.calculate_profile_for_group(selected, poi_types)

In [9]:
selected

osm_node_id,time,cost,n_transfers,human_readable_time,start_id_hex,osm_id,lat,long,nearest_osm_node,dist_to_nearest,poi_type,target_id_osm,Education,Shops,Parks,Health,Grocery,Sustenance,Banks
i64,i64,i64,i64,str,str,u64,f64,f64,u64,f64,str,i64,i8,i8,i8,i8,i8,i8,i8
3668330662,30821,0,0,"""08:33:41""","""881fa18ab5fffff""",11763743784,50.897879,7.0766994,3668330662,16.196938,"""Education""",3668330662,1,0,0,0,0,0,0
366310180,31313,0,0,"""08:41:53""","""881fa18ab5fffff""",6885637629,50.906934,7.1549169,366310180,3.859437,"""Parks""",366310180,0,0,1,0,0,0,0
1458679913,31265,0,0,"""08:41:05""","""881fa18ab5fffff""",6481363798,50.894217,7.0719118,1458679913,18.955936,"""Health""",1458679913,0,0,0,1,0,0,0
7309941565,30892,0,0,"""08:34:52""","""881fa18ab5fffff""",3423722201,50.920304,7.0975108,7309941565,13.807823,"""Banks""",7309941565,0,0,0,0,0,0,1
304146002,30899,0,0,"""08:34:59""","""881fa18ab5fffff""",358373468,50.904801,7.1468239,304146002,16.337421,"""Sustenance""",304146002,0,0,0,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
11697309354,30817,0,0,"""08:33:37""","""881fa18ab5fffff""",4204733130,50.894208,7.0783708,11697309354,16.462961,"""Shops""",11697309354,0,1,0,0,0,0,0
1727812560,31169,0,0,"""08:39:29""","""881fa18ab5fffff""",2187011541,50.897078,7.0713095,1727812560,14.455975,"""Grocery""",1727812560,0,0,0,0,1,0,0
446940874,30745,0,0,"""08:32:25""","""881fa18ab5fffff""",446784373,50.898403,7.082495,446940874,20.942789,"""Shops""",446940874,0,1,0,0,0,0,0


In [11]:
selected.group_by("poi_type").agg(pl.col("human_readable_time").min())

poi_type,human_readable_time
str,str
"""Health""","""08:01:15"""
"""Banks""","""08:11:53"""
"""Shops""","""08:06:11"""
"""Education""","""08:10:19"""
"""Parks""","""08:02:35"""
"""Sustenance""","""08:00:00"""
"""Grocery""","""08:03:54"""


In [4]:
geo_data.pois.group_by("poi_type").len()

poi_type,len
str,u32
"""Grocery""",2911
"""Health""",1821
"""Education""",545
"""Banks""",732
"""Parks""",16970
"""Shops""",5990
"""Sustenance""",4004
